# Vani-Kanoon Legal LLM - Fixed Version

**IMPORTANT:** After running cell 1, go to **Runtime → Restart session**, then run from cell 2.

In [ ]:
# Cell 1: Install (run once, then restart runtime)
!pip install -q transformers==4.38.0 datasets accelerate peft==0.9.0
print("\n" + "="*50)
print("NOW GO TO: Runtime → Restart session")
print("Then run from Cell 2 onwards (skip this cell)")
print("="*50)

In [ ]:
# Cell 2: Imports
import json
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Load data
import os
for p in ["/kaggle/input/vani-kanoon-training/training_data.json", "training_data.json"]:
    if os.path.exists(p):
        with open(p, 'r') as f:
            raw_data = json.load(f)
        print(f"Loaded {len(raw_data)} examples from {p}")
        break

def fmt(ex):
    i, inp, o = ex.get('instruction',''), ex.get('input',''), ex.get('output','')
    return {"text": f"<|user|>\n{i}\n{inp}\n<|assistant|>\n{o}" if inp else f"<|user|>\n{i}\n<|assistant|>\n{o}"}

dataset = Dataset.from_list([fmt(ex) for ex in raw_data])
print(f"Dataset: {len(dataset)}")

In [ ]:
# Cell 4: Load model
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model.config.use_cache = False

print(f"Model: {model.num_parameters():,} params")
print(f"GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB used")

In [ ]:
# Cell 5: Add LoRA
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
))
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 6: Tokenize
def tok(ex):
    r = tokenizer(ex["text"], truncation=True, max_length=512, padding="max_length")
    r["labels"] = r["input_ids"].copy()
    return r

tokenized = dataset.map(tok, batched=True, remove_columns=["text"])
print(f"Ready: {len(tokenized)}")

In [ ]:
# Cell 7: Train
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./out", num_train_epochs=5,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        learning_rate=3e-4, fp16=True, logging_steps=10,
        save_strategy="epoch", report_to="none"
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Training...")
trainer.train()
print("Done!")

In [ ]:
# Cell 8: Save
model.save_pretrained("./vani-legal")
tokenizer.save_pretrained("./vani-legal")

import shutil
shutil.make_archive("vani-legal", 'zip', "./vani-legal")
print("Saved: vani-legal.zip")

In [ ]:
# Cell 9: Test
def ask(q):
    inp = tokenizer(f"<|user|>\n{q}\n<|assistant|>\n", return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=200, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("<|assistant|>")[-1].strip()

print(ask("What is punishment for murder under BNS?"))